In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!pip install llama-index llama-index-llms-openai llama-index-embeddings-openai

In [ ]:
!pip install llama-index llama-index-llms-huggingface llama-index-embeddings-huggingface transformers accelerate bitsandbytes

In [ ]:
!pip install llama-index-readers-file

In [ ]:
import os


folder_name = "movie_data"
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"Directory '{folder_name}' created!")


sample_text = """
Inception is a 2010 science fiction action film written and directed by Christopher Nolan.
The film stars Leonardo DiCaprio as a professional thief who steals information by infiltrating the subconscious of his targets.
It won four Academy Awards and is known for its complex plot involving dreams within dreams.
"""

with open(f"{folder_name}/inception.txt", "w") as f:
    f.write(sample_text)
    print("Sample movie data 'inception.txt' created!")

In [ ]:
import torch
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from transformers import BitsAndBytesConfig


Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)


quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)


Settings.llm = HuggingFaceLLM(
    model_name="microsoft/Phi-3-mini-4k-instruct",
    tokenizer_name="microsoft/Phi-3-mini-4k-instruct",
    context_window=4096,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.0},
    model_kwargs={"quantization_config": quantization_config},
    device_map="auto",
)


documents = SimpleDirectoryReader("./movie_data").load_data()
index = VectorStoreIndex.from_documents(documents)

query_engine = index.as_query_engine(similarity_top_k=3)

def run_ntcir_query(question):
    prompt = f"Based on the context, answer the question. Provide a CONFIDENCE score (0-100) and bulleted NUGGETS. \nQuestion: {question}"
    response = query_engine.query(prompt)
    return response

# Test
print(run_ntcir_query("Tell me about the movie Inception."))

In [ ]:
import torch
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from transformers import BitsAndBytesConfig

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")


quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)


Settings.llm = HuggingFaceLLM(
    model_name="microsoft/Phi-3-mini-4k-instruct",
    tokenizer_name="microsoft/Phi-3-mini-4k-instruct",
    model_kwargs={"quantization_config": quantization_config},
    device_map="auto",
)

documents = SimpleDirectoryReader("./movie_data").load_data()
index = VectorStoreIndex.from_documents(documents)


from llama_index.core import PromptTemplate

def ask_ntcir(question):
    query_engine = index.as_query_engine(similarity_top_k=3)


    prompt_str = (
    "<|system|>\n"
    "You are a movie researcher for NTCIR-19.\n"
    "You MUST answer ONLY using the provided context.\n"
    "If the context does not contain the answer, you MUST respond with 'I don't know'.\n"
    "Do NOT guess and do NOT use outside knowledge.\n"
    "If the movie mentioned in the question is not present in the context, say 'I don't know'.\n"
    "When you say 'I don't know', set CONFIDENCE to 0.\n"
    "<|end|>\n"

    "<|user|>\n"
    "Context:\n{context_str}\n\n"
    f"Question: {question}\n\n"
    "Follow this response format exactly:\n"
    "ANSWER: [direct answer OR 'I don't know']\n"
    "CONFIDENCE: [0-100]\n"
    "NUGGETS:\n"
    "- [supporting facts from the context]\n"
    "- [another supporting fact]\n\n"
    "If the context does not contain the answer, respond exactly with:\n"
    "ANSWER: I don't know\n"
    "CONFIDENCE: 0\n"
    "NUGGETS:\n"
    "- The provided documents do not contain the requested information.\n"
    "<|end|>\n"

    "<|assistant|>\n"
)

    text_qa_template = PromptTemplate(prompt_str)


    query_engine.update_prompts(
        {"response_synthesizer:text_qa_template": text_qa_template}
    )

    response = query_engine.query(question)
    return response


print(ask_ntcir("What is the main plot of the Inception movie in the database?"))

print(ask_ntcir("What is the main plot of the nadiya ke par movie in the database?"))

In [ ]:
import pandas as pd

def run_batch_evaluation(questions_list):
    all_results = []

    print(f"Starting evaluation for {len(questions_list)} questions...")

    for i, q in enumerate(questions_list):
        print(f"Processing Q{i+1}: {q}")
        response = ask_ntcir(q)

        all_results.append({
            "question": q,
            "system_output": str(response)
        })


    df = pd.DataFrame(all_results)
    df.to_csv("ntcir_r2c2_results.csv", index=False)
    print("Done! Results saved to 'ntcir_r2c2_results.csv'")
    return df

test_questions = [
    "Who directed the movie Inception?",
    "When was Inception released?",
    "What awards did Inception win?"
]

results_df = run_batch_evaluation(test_questions)

worked on 100 movies data files

In [ ]:
from datasets import load_dataset
import os
import re

print("Downloading 100 movie records from Hugging Face...")
dataset = load_dataset("Coder-Dragon/wikipedia-movies", split="train[:100]")

output_dir = "./movie_data"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

for i, item in enumerate(dataset):
    title = item['Title']
    plot = item['Plot']


    safe_title = re.sub(r'[^\w\s]', '', title).strip().replace(' ', '_')
    filename = f"{output_dir}/{safe_title}.txt"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"MOVIE TITLE: {title}\n")
        f.write(f"STORY PLOT: {plot}")

print(f"Success! Your library now contains {len(os.listdir(output_dir))} movie files.")

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter

documents = SimpleDirectoryReader("./movie_data").load_data()

splitter = SentenceSplitter(
    chunk_size=200,
    chunk_overlap=50
)

nodes = splitter.get_nodes_from_documents(documents)

index = VectorStoreIndex(nodes)

index.storage_context.persist(persist_dir="./storage")

print("Index built with chunking and saved to ./storage")

In [ ]:

print("Available Movies (First 5):", os.listdir("./movie_data")[:5])

response = ask_ntcir("Who is the main character in Alice in Wonderland?")
print(response)

In [ ]:
retriever = index.as_retriever(similarity_top_k=3)

nodes = retriever.retrieve("Who is the main character in Alice in Wonderland?")

for node in nodes:
    print("Retrieved Text:")
    print(node.text)
    print("Source:", node.metadata)
    print("------------")

In [ ]:
import re

def generate_ntcir_run(questions_dict, run_name="output_ntcir"):
    run_output = []

    for topic_id, question in questions_dict.items():
        print(f"Processing Topic {topic_id}...")
        raw_response = ask_ntcir(question)
        response_text = str(raw_response)

        answer_match = re.search(r"ANSWER:\s*(.*)", response_text, re.IGNORECASE)
        answer = answer_match.group(1).strip() if answer_match else "I don't know"

        conf_match = re.search(r"CONFIDENCE:\s*(\d+)", response_text)
        confidence = conf_match.group(1) if conf_match else "0"

        nuggets = re.findall(r"-\s*(.*)", response_text)

        run_output.append(f"<{topic_id}>")
        run_output.append(f"{answer};{confidence}")


        for i, nugget in enumerate(nuggets[:10]):
            run_output.append(f"{i+1};LOCAL;1;{nugget.strip()}")

        run_output.append(f"</{topic_id}>")

    with open(f"{run_name}.txt", "w") as f:
        f.write("\n".join(run_output))

    print(f"Submission file created: {run_name}.txt")

my_topics = {
    "0001": "What is the plot of Alice in Wonderland?",
    "0002": "Who is the main character in Pocahontas?"
}

generate_ntcir_run(my_topics)

In [ ]:
def test_system_modesty():
    print("--- 🛡️ Starting Modesty Stress Test ---")

    tricky_question = "What happens in the movies chand ke par chalo'?"

    print(f"Testing with OOD Question: {tricky_question}")
    response = ask_ntcir(tricky_question)

    response_text = str(response)
    conf_match = re.search(r"CONFIDENCE:\s*(\d+)", response_text)
    confidence = int(conf_match.group(1)) if conf_match else 100

    print(f"\nModel Output:\n{response_text}")

    if confidence < 30:

        print("\n TEST PASSED: The system is modest. It recognized it lacked context.")
    else:
        print("\n TEST FAILED: The system is 'overconfident'. You may need to tune the prompt.")

test_system_modesty()

**for large data on hugging face**

In [ ]:
from datasets import load_dataset
import os
import re

# 1. Load the dataset
print("Downloading large movie dataset...")
dataset = load_dataset("AIatMongoDB/embedded_movies", split="train")

# 2. Setup your movie_data folder
output_dir = "./movie_data_large"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 3. Save as individual files (NTCIR Style)
print("Processing and saving files...")
for i, item in enumerate(dataset):
    # Skip records with missing essential data
    if not item.get('fullplot') or not item.get('title'):
        continue

    title = item['title']
    plot = item['fullplot']

    # FIX: Use 'or []' to handle None values gracefully
    directors = item.get('directors') or []
    cast = item.get('cast') or []

    # Sanitize title for filename
    safe_title = re.sub(r'[^\w\s]', '', title).strip().replace(' ', '_')
    filename = f"{output_dir}/{safe_title}_{i}.txt"

    try:
        with open(filename, "w", encoding="utf-8") as f:
            f.write(f"MOVIE TITLE: {title}\n")
            f.write(f"DIRECTORS: {', '.join(directors)}\n")
            f.write(f"CAST: {', '.join(cast)}\n")
            f.write(f"STORY PLOT: {plot}")
    except Exception as e:
        print(f"Skipping {title} due to error: {e}")

print(f"Success! Your library now contains {len(os.listdir(output_dir))} movie files.")

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext

# Load the new large directory
documents = SimpleDirectoryReader("./movie_data_large").load_data()

# Build the index (This will take 2-5 minutes for 1,500 documents)
index = VectorStoreIndex.from_documents(documents)

# Persist the large index so you don't have to rebuild it
index.storage_context.persist(persist_dir="./storage_large")
print("Large-scale index complete and saved!")

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage

# 1. Point to the folder where you saved the large index
storage_context = StorageContext.from_defaults(persist_dir="./storage_large")

# 2. Load the index back into memory
# Note: You still need your 'Settings.llm' and 'Settings.embed_model' defined
index = load_index_from_storage(storage_context)

# 3. Create the query engine (The Librarian)
query_engine = index.as_query_engine(similarity_top_k=3)

print("✅ System Re-initialized from Saved Data!")

In [ ]:
# Test it out!
test_query = "Who are the cast members of the movie 'The Perils of Pauline'?"
response = query_engine.query(test_query)

print(f"User: {test_query}")
print(f"AI: {response}")

In [ ]:
def ask_ntcir_large(question, engine):
    """
    Accepts a question and a specific query_engine (the large-scale one).
    Returns the NTCIR-formatted response.
    """
    prompt = f"""
    You are an NTCIR-19 R2C2 research assistant.
    Based ONLY on the provided context, answer the question.

    FORMAT:
    ANSWER: [A concise 1-sentence answer]
    CONFIDENCE: [0-100 score based on context availability]
    NUGGETS:
    - [Key fact 1]
    - [Key fact 2]

    QUESTION: {question}
    """

    # We pass the question to the specific engine provided
    response = engine.query(prompt)
    return response

# Use the 'query_engine' we just loaded from './storage_large'
large_scale_response = ask_ntcir_large(
    "What is the main plot of 'Baseball and Bloomers'?",
    engine=query_engine
)

print(large_scale_response)

In [ ]:
# This shows you exactly which files the 'Librarian' is looking at
retrieved_nodes = query_engine.retrieve("Baseball and Bloomers")
for node in retrieved_nodes:
    print(f"Source Found: {node.node.metadata['file_name']}")
    print(f"Similarity Score: {node.score:.4f}\n")

In [ ]:
from llama_index.core import PromptTemplate

def run_modesty_test(query_engine):
    test_cases = [
        {
            "type": "In-Distribution (Should be High Confidence)",
            "query": "Who is the main character in the 1914 film 'The Perils of Pauline'?"
        },
        {
            "type": "Out-of-Distribution (Should be ZERO Confidence)",
            "query": "What happens at the end of the movie 'Deadpool & Wolverine'?"
        }
    ]

    print("--- STARTING MODESTY TEST ---\n")

    for case in test_cases:
        print(f"TEST TYPE: {case['type']}")
        print(f"QUERY: {case['query']}")

        # FIX: Wrap the string in PromptTemplate and use {context_str} and {query_str}
        modesty_prompt_str = """
        Instructions: Answer based ONLY on the provided context.
        If the information is not in the context, set CONFIDENCE to 0 and ANSWER to 'Information not found'.

        FORMAT:
        ANSWER: [Response]
        CONFIDENCE: [0-100]

        CONTEXT: {context_str}

        QUESTION: {query_str}
        """
        modesty_template = PromptTemplate(modesty_prompt_str)

        # Update the engine with the proper Template Object
        query_engine.update_prompts({
            "response_synthesizer:text_qa_template": modesty_template
        })

        response = query_engine.query(case['query'])
        print(f"MODEL RESPONSE:\n{response}\n")
        print("-" * 30)

# Run the fixed test
run_modesty_test(query_engine)

Implimenting using Langchain and gimini-2.5 flash and pinecone vector database

Now using this because Gemini cause a single point of failure when pushing data in pinecone, so we ar currently using-all-mpnet-base-v2 transformer model

In [ ]:
pip install -U langgraph langchain-pinecone langchain-google-genai pinecone-client

In [ ]:
!pip install -U langchain-community langchain-text-splitters

In [ ]:
!pip install sentence-transformers

In [ ]:
!pip install -U langchain-huggingface sentence-transformers

In [ ]:
!pip install -U langchain-google-genai

In [ ]:
import os
import time
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

# 1. SETUP PINECONE KEY
os.environ["PINECONE_API_KEY"] = "your key here"  # Replace with your actual Pinecone API key
P_KEY = os.environ["PINECONE_API_KEY"]
index_name = "movie-index"

# 2. LOAD & SPLIT DATA
print("📂 Loading 1,500 movies from storage...")
if not os.path.exists('./movie_data_large'):
    print("❌ Error: Directory './movie_data_large' not found!")
else:
    loader = DirectoryLoader('./movie_data_large', glob="./*.txt", loader_cls=TextLoader)
    raw_documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    docs = text_splitter.split_documents(raw_documents)
    print(f"✂️ Created {len(docs)} chunks.")

    # 3. INITIALIZE LOCAL EMBEDDINGS
    # No more 404/429 errors—this runs on your CPU/GPU
    print("🤖 Initializing Local Transformer (all-mpnet-base-v2)...")
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

    # 4. UPLOAD TO PINECONE
    print(f"🚀 Starting Upload to Pinecone Index: {index_name}...")
    BATCH_SIZE = 100 # We can go much faster now!

    try:
        # Initialize with first batch
        initial_batch = docs[0:BATCH_SIZE]
        print("Syncing Batch 1 (Creating Connection)...")
        vectorstore = PineconeVectorStore.from_documents(
            initial_batch,
            embeddings,
            index_name=index_name,
            pinecone_api_key=P_KEY
        )

        # Remaining Batches
        for i in range(BATCH_SIZE, len(docs), BATCH_SIZE):
            batch = docs[i : i + BATCH_SIZE]
            print(f"Syncing Batch {i//BATCH_SIZE + 1} of {len(docs)//BATCH_SIZE + 1}...")
            vectorstore.add_documents(batch)
            # Short sleep to prevent Pinecone rate limits (not Google)
            time.sleep(1)

        print("✅ SUCCESS: 1,500 movies are now fully indexed in Pinecone!")

    except Exception as e:
        print(f"❌ System Failure: {e}")

In [ ]:
import os
import requests
import time
from langchain_pinecone import PineconeVectorStore

# 1. SETUP
# Make sure you use a NEW API Key if the old one was marked as leaked!
G_KEY = input("Enter your NEW Gemini API Key: ")
index_name = "movie-index"
vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)

# 2. RETRIEVAL (Top-1 Only)
query = "Suggest a movie where a character struggles with their own modesty or overconfidence."
print(f"🔍 Searching for the absolute best match...")
retrieved_docs = vectorstore.similarity_search(query, k=1)

# Compress: Take only the first 800 characters
context_summary = retrieved_docs[0].page_content[:800]

# 3. DIRECT CALL TO FLASH-LITE (Highest Free Quota)
def call_gemini_free_tier(prompt_text, api_key):
    # Using the 'lite' model for maximum free-tier throughput
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent?key={api_key}"
    payload = {"contents": [{"parts": [{"text": prompt_text}]}]}

    # Try once, wait 65 seconds if it fails (Free tier reset window)
    for attempt in range(2):
        response = requests.post(url, json=payload)
        if response.status_code == 200:
            return response.json()['candidates'][0]['content']['parts'][0]['text']
        elif response.status_code == 429:
            print("⏳ Free tier busy. Waiting 65 seconds for quota reset...")
            time.sleep(65)
        else:
            return f"❌ Error {response.status_code}: {response.text}"
    return "❌ Still busy. Try again in a few minutes."

# 4. COMPACT PROMPT
ntcir_prompt = f"""
NTCIR-19 Analysis:
MOVIE PLOT: {context_summary}
USER QUERY: {query}
Does this movie fit? Explain why or why not briefly.
"""

print("🧠 Generating Response...")
result = call_gemini_free_tier(ntcir_prompt, G_KEY)

print("-" * 30)
print("🎬 NTCIR-19 PREDICTION RESULT:")
print("-" * 30)
print(result)

In [ ]:
!pip install groq

In [ ]:
import os
from groq import Groq
from langchain_pinecone import PineconeVectorStore

# 1. AUTHENTICATION
GROQ_API_KEY = "your key here"  # Replace with your actual Groq API key
# (Ensure your Pinecone key is still set in the environment)

# 2. INITIALIZE RETRIEVER
print("🤖 Connecting to Pinecone Index...")
index_name = "movie-index"
vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)

# 3. SEARCH
query = "Suggest a movie where a character struggles with their own modesty or overconfidence."
print(f"🔍 Searching Pinecone Index...")
retrieved_docs = vectorstore.similarity_search(query, k=3)
context_text = "\n\n".join([f"Movie Plot: {doc.page_content}" for doc in retrieved_docs])

# 4. GROQ INFERENCE (Updated to Llama 3.3)
client = Groq(api_key=GROQ_API_KEY)

print("🧠 Generating NTCIR Response via Groq (Llama 3.3-70B)...")
try:
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": "You are a professional movie critic. Analyze the plot provided. Focus on themes of modesty vs. overconfidence."
            },
            {
                "role": "user",
                "content": f"CONTEXT:\n{context_text}\n\nUSER REQUEST: {query}"
            }
        ],
        # UPDATED MODEL ID FOR 2026
        model="llama-3.3-70b-versatile",
        temperature=0.1,
    )

    print("-" * 30)
    print("🎬 NTCIR-19 PREDICTION RESULT:")
    print("-" * 30)
    print(chat_completion.choices[0].message.content)

except Exception as e:
    print(f"❌ Execution Error: {e}")

In [ ]:
import os
from groq import Groq
from langchain_pinecone import PineconeVectorStore

# 1. SETUP
GROQ_API_KEY = "your key here"  # Replace with your actual Groq API key
client = Groq(api_key=GROQ_API_KEY)
index_name = "movie-index"
vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)

# 2. RETRIEVE
query = "Suggest a movie where a character struggles with their own modesty or overconfidence."
retrieved_docs = vectorstore.similarity_search(query, k=3)
context_text = "\n\n".join([f"SOURCE_{i+1}: {doc.page_content}" for i, doc in enumerate(retrieved_docs)])

# 3. STRUCTURED INFERENCE
print("🧠 Formatting NTCIR-19 Official Response...")
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": """You are an NTCIR-19 Evaluator. Provide answers in the following STATED format:

            ### [MOVIE TITLE]
            - **Theme Focus**: (Modesty or Overconfidence)
            - **Character Analysis**: (2-3 sentences on the struggle)
            - **Source Grounding**: (Quote a specific detail from the provided context)
            - **Confidence Score**: (0.0 to 1.0 based on how well the context matches)

            If the context does not support the answer, return 'STATUS: INSUFFICIENT DATA'."""
        },
        {
            "role": "user",
            "content": f"CONTEXT FROM VECTOR DB:\n{context_text}\n\nUSER QUERY: {query}"
        }
    ],
    model="llama-3.3-70b-versatile",
    temperature=0, # Deterministic for NTCIR
)

print("-" * 30)
print("🎬 OFFICIAL NTCIR-19 OUTPUT:")
print("-" * 30)
print(chat_completion.choices[0].message.content)

In [ ]:
import os
from groq import Groq
from langchain_pinecone import PineconeVectorStore

# 1. SETUP
# Use your existing Groq Key
GROQ_API_KEY = "your key here"  # Replace with your actual Groq API key
client = Groq(api_key=GROQ_API_KEY)
index_name = "movie-index"

# 2. INITIALIZE VECTOR STORE
# Using the 768-D HuggingFace embeddings from your setup
vectorstore = PineconeVectorStore(index_name=index_name, embedding=embeddings)

# 3. THE COMPLEX QUERY
complex_query = "Find a movie where a character's greatest strength becomes their ultimate undoing or tragic flaw."

print(f"🔍 Searching 1,500 movies for complex narrative patterns...")
# We pull Top-3 to give the LLM enough 'thematic' variety to choose the best fit
retrieved_docs = vectorstore.similarity_search(complex_query, k=3)

context_text = ""
for i, doc in enumerate(retrieved_docs):
    context_text += f"SOURCE_{i+1}:\n{doc.page_content}\n---\n"

# 4. STRUCTURED INFERENCE (NTCIR-19 Format)
print("🧠 Analyzing Subtext with Llama 3.3-70B...")
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": """You are an NTCIR-19 Evaluator specializing in Literary Archetypes.
            Identify a movie from the context where a character experiences a 'Tragic Flaw' (Hamartia).

            Format:
            ### [MOVIE TITLE]
            - **The Strength**: (What was the positive trait?)
            - **The Undoing**: (How did that trait cause the downfall?)
            - **Evidence**: (Reference a specific plot point from the context)
            - **Confidence Score**: (0.0 - 1.0)"""
        },
        {
            "role": "user",
            "content": f"CONTEXT:\n{context_text}\n\nQUERY: {complex_query}"
        }
    ],
    model="llama-3.3-70b-versatile",
    temperature=0,
)

print("-" * 30)
print("🎬 COMPLEX QUERY RESULT:")
print("-" * 30)
print(chat_completion.choices[0].message.content)

In [ ]:
!pip install langgraph

In [ ]:
import operator
from typing import Annotated, List, TypedDict
from langgraph.graph import StateGraph, END
from groq import Groq

# 1. DEFINE THE STATE
# This acts as the "shared memory" between nodes
class GraphState(TypedDict):
    query: str
    context: str
    prediction: str
    relevance_score: str # 'relevant' or 'irrelevant'

# 2. DEFINE THE NODES (The Workers)
def retrieve_node(state: GraphState):
    print("---RETRIEVING FROM PINECONE---")
    # Using your existing vectorstore logic
    docs = vectorstore.similarity_search(state['query'], k=3)
    return {"context": "\n".join([d.page_content for d in docs])}

def grade_node(state: GraphState):
    print("---GRADING RELEVANCE---")
    # We ask Llama 3.3 to act as a judge
    grader_prompt = f"Context: {state['context']}\nQuery: {state['query']}\nIs this context relevant? Answer 'yes' or 'no'."
    # (Simplified for example - call your Groq function here)
    return {"relevance_score": "yes"} 

def generate_node(state: GraphState):
    print("---GENERATING FINAL ANSWER---")
    # Final production of the NTCIR response
    return {"prediction": "The movie matching your query is..."}

# 3. DEFINE THE EDGES (The Logic Flow)
def decide_to_generate(state: GraphState):
    if state["relevance_score"] == "yes":
        return "generate"
    else:
        return "rewrite" # You could add a rewrite node here!

# 4. BUILD THE GRAPH
workflow = StateGraph(GraphState)

# Add Nodes
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("grade", grade_node)
workflow.add_node("generate", generate_node)

# Connect them
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade")
workflow.add_conditional_edges(
    "grade",
    decide_to_generate,
    {
        "generate": "generate",
        "rewrite": "retrieve", # Loop back to search again if irrelevant
    },
)
workflow.add_edge("generate", END)

# Compile
app = workflow.compile()

#Adding Hybrid Search

In [ ]:
!pip install pinecone-text

In [ ]:

from pinecone_text.sparse import BM25Encoder
from langchain_community.retrievers import PineconeHybridSearchRetriever

# 1. Initialize BM25 (Keyword Encoder)
bm25 = BM25Encoder().default()

# 2. Fit the encoder on your movie titles/plots
# This 'teaches' the model which words are unique to your 1,500 movies
movie_texts = [doc.page_content for doc in all_docs]
bm25.fit(movie_texts)

# 3. Create the Hybrid Retriever
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings, 
    sparse_encoder=bm25, 
    index=index
)

# 4. Perform a Hybrid Search
# This will now find "Iron Man" even if the vector meaning is slightly off
query = "Suggest a movie with a secret twin twist"
results = retriever.get_relevant_documents(query)